# Solution: Challenge Week 08

## Libraries and settings

In [ ]:
# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Libraries
import os
import requests
import pandas as pd
import matplotlib.pyplot as plt
import folium
from folium.plugins import MarkerCluster

# Show current working directory
print(os.getcwd())

## Read the data from opendata.swiss

The dataset is published by the Swiss Federal Office of Energy (SFOE/BFE) in OICP format.
The JSON structure is **two-level**: the top level contains one entry per operator
(`OperatorID`, `OperatorName`, `EVSEDataRecord`), and `EVSEDataRecord` is the list
of individual charging stations. We flatten both levels into a single DataFrame.

In [ ]:
# Step 1: Retrieve the data URL via the opendata.swiss CKAN API
ckan_url = 'https://opendata.swiss/api/3/action/package_show?id=ladestationen-fuer-elektroautos'
response = requests.get(ckan_url)
resources = response.json()['result']['resources']

# Use the first JSON resource (static EVSE data)
json_resources = [r for r in resources if str(r.get('format', '')).upper() == 'JSON']
data_url = json_resources[0].get('download_url', json_resources[0].get('url'))
print(f'Downloading data from:\n  {data_url}\n')

# Step 2: Download the JSON file
data_response = requests.get(data_url)
raw_data = data_response.json()

# The JSON has one top-level key 'EVSEData' containing a list of operator objects
operator_list = raw_data['EVSEData']
print(f'Number of operators in the dataset : {len(operator_list)}')
print(f'Keys in each operator entry        : {list(operator_list[0].keys())}')

In [ ]:
# Step 3: Flatten the two-level structure into one row per charging station
#
#   EVSEData (list of 38 operators)
#     └── EVSEDataRecord (list of N stations per operator)  <-- the actual stations
#
# Strategy: iterate over operators, inject OperatorID + OperatorName into every
# station record, then build a flat DataFrame with pd.json_normalize.

all_records = []
for operator in operator_list:
    op_id   = operator['OperatorID']
    op_name = operator['OperatorName']
    for station in operator['EVSEDataRecord']:
        station['OperatorID']   = op_id
        station['OperatorName'] = op_name
        all_records.append(station)

df_raw = pd.json_normalize(all_records)

print(f'Total charging stations : {len(df_raw)}')
print(f'Number of columns       : {len(df_raw.columns)}')
df_raw[['OperatorID', 'OperatorName', 'EvseID', 'GeoCoordinates.Google']].head()

## Count the number of operators

In [ ]:
# Count charging stations per operator
df_operators = (
    df_raw.groupby(['OperatorID', 'OperatorName'])
    .size()
    .reset_index(name='Number_of_Stations')
    .sort_values('Number_of_Stations', ascending=False)
    .reset_index(drop=True)
)

print(f'Total number of unique operators : {len(df_operators)}')
print(f'Total number of charging stations: {df_operators["Number_of_Stations"].sum()}\n')
df_operators

In [ ]:
# Horizontal bar chart: Top 15 operators by number of charging stations
top15 = df_operators.head(15).sort_values('Number_of_Stations')

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(top15['OperatorName'], top15['Number_of_Stations'], color='steelblue')
ax.bar_label(bars, padding=4, fontsize=9)
ax.set_title('Top 15 Operators by Number of Charging Stations (Switzerland)', fontsize=12)
ax.set_xlabel('Number of Charging Stations')
ax.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Derive all data from operator 'eCarUp'

In [ ]:
# Filter by OperatorName containing 'eCarUp' (case-insensitive)
df_ecarup = df_raw[
    df_raw['OperatorName'].str.lower().str.contains('ecarup', na=False)
].copy()

print(f"OperatorID   : {df_ecarup['OperatorID'].unique().tolist()}")
print(f"OperatorName : {df_ecarup['OperatorName'].unique().tolist()}")
print(f"Stations     : {len(df_ecarup)}")
print(f"Shape        : {df_ecarup.shape}\n")

# Show selected columns
df_ecarup[[
    'EvseID', 'OperatorID', 'OperatorName',
    'Address.Street', 'Address.City', 'Address.PostalCode',
    'GeoCoordinates.Google'
]].head(10)

In [ ]:
# Parse GeoCoordinates.Google (format: 'lat lon') into separate numeric columns
geo = df_ecarup['GeoCoordinates.Google'].str.split(' ', expand=True)
df_ecarup['lat'] = pd.to_numeric(geo[0], errors='coerce')
df_ecarup['lon'] = pd.to_numeric(geo[1], errors='coerce')

df_ecarup_geo = df_ecarup.dropna(subset=['lat', 'lon'])

print(f'Stations with valid coordinates: {len(df_ecarup_geo)} / {len(df_ecarup)}')
print(f'Latitude  range: {df_ecarup_geo["lat"].min():.4f} – {df_ecarup_geo["lat"].max():.4f}')
print(f'Longitude range: {df_ecarup_geo["lon"].min():.4f} – {df_ecarup_geo["lon"].max():.4f}')

## Plot eCarUp charging stations on a map

In [ ]:
# Centre map on mean coordinates of eCarUp stations
centre_lat = df_ecarup_geo['lat'].mean()
centre_lon = df_ecarup_geo['lon'].mean()

# Create folium map
m = folium.Map(
    location=[centre_lat, centre_lon],
    zoom_start=8,
    tiles='CartoDB positron'
)

# MarkerCluster for cleaner display when stations overlap
cluster = MarkerCluster(name='eCarUp Charging Stations').add_to(m)

# Add one circle marker per station
for _, row in df_ecarup_geo.iterrows():
    # Build popup text from address
    popup_text = (
        f"<b>{row.get('EvseID', '')}</b><br>"
        f"{row.get('Address.Street', '')} <br>"
        f"{row.get('Address.PostalCode', '')} {row.get('Address.City', '')}"
    )
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=7,
        color='#c0392b',
        fill=True,
        fill_color='#e74c3c',
        fill_opacity=0.85,
        popup=folium.Popup(popup_text, max_width=220),
        tooltip=row.get('Address.City', 'eCarUp')
    ).add_to(cluster)

# Title overlay
title_html = '''
<div style="position:fixed;top:10px;left:50%;transform:translateX(-50%);
            background:white;padding:7px 16px;border-radius:6px;
            box-shadow:2px 2px 6px rgba(0,0,0,.3);font-size:14px;
            font-family:Arial;z-index:9999;">
    <b>eCarUp Charging Stations – Switzerland</b>
</div>
'''
m.get_root().html.add_child(folium.Element(title_html))
folium.LayerControl().add_to(m)

# Save to html
map_path = './ecarup_charging_stations_map.html'
m.save(map_path)

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')